# Phase 3 — Explainability Benchmark (RQ2)

**Objective:** Quantify how XAI fidelity, localization and stability degrade under image-quality changes.

**XAI methods:**
- **Grad-CAM** — ResNet-50 + EfficientNet-B3 (CNNs)
- **SHAP** (KernelSHAP on superpixels) — all three models
- **Attention rollout** — ViT-Base only

**Metrics:**
- **Fidelity**: Deletion AUC (lower = better) and Insertion AUC (higher = better)
- **Stability**: Spearman rank-correlation between heatmaps on clean vs degraded image
- **Localization**: IoU vs ground-truth lesion masks (skipped automatically if masks unavailable)

**Outputs (under `Drive/Thesis/results/phase3_xai_benchmark/`):**
- `metrics/xai_results.csv` — long format (image, model, method, degradation, level, deletion_auc, insertion_auc, stability, iou)
- `samples/heatmap_<model>_<method>_<id>.png` — qualitative figures
- `plots/<metric>_vs_degradation_<type>.png`

> **Note on cost:** SHAP is the slowest. Reduce `N_EXPLAIN` if your Colab session is short.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q timm captum shap scikit-image

## Config

In [ ]:
from pathlib import Path
import torch

DRIVE_ROOT      = Path('/content/drive/MyDrive/Thesis')
RESULTS_ROOT    = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
PHASE_DIR       = RESULTS_ROOT / 'phase3_xai_benchmark'
PHASE1_DIR      = RESULTS_ROOT / 'phase1_data_engineering'
PHASE2_DIR      = RESULTS_ROOT / 'phase2_model_benchmarking'
for sub in ('metrics', 'plots', 'samples', 'logs'):
    (PHASE_DIR / sub).mkdir(parents=True, exist_ok=True)

LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'
APTOS_IMAGES = next(p for p in APTOS_DIR.rglob('train_images') if p.is_dir())
PRISTINE_CSV = PHASE1_DIR / 'metrics' / 'pristine_split.csv'

NUM_CLASSES = 5
IMAGE_SIZE  = 224
MODEL_NAMES = ('resnet50', 'efficientnet_b3', 'vit_base')
TIMM_NAMES  = {'resnet50': 'resnet50', 'efficientnet_b3': 'efficientnet_b3',
               'vit_base': 'vit_base_patch16_224'}
DEGRADATION_TYPES  = ('blur', 'exposure', 'noise')
DEGRADATION_LEVELS = ('low', 'mid', 'high')

# How many test images we explain. Keep small — XAI is expensive.
N_EXPLAIN     = 20
DELETION_STEPS = 30
SHAP_N_SAMPLES = 60   # KernelSHAP — bump to 100+ for final results

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Reload trained models

In [ ]:
import timm
import torch.nn as nn

def build_model(name):
    m = timm.create_model(TIMM_NAMES[name], pretrained=False, num_classes=NUM_CLASSES)
    m._dr_arch = name
    return m

def load_model(name):
    m = build_model(name).to(DEVICE)
    state = torch.load(CHECKPOINTS_DIR / f'{name}_best.pt', map_location=DEVICE)
    m.load_state_dict(state['state_dict']); m.eval()
    return m

def get_xai_target_layer(model):
    arch = model._dr_arch
    if arch == 'resnet50':        return model.layer4[-1]
    if arch == 'efficientnet_b3': return model.blocks[-1]
    if arch == 'vit_base':        return model.blocks[-1].norm1
    raise ValueError(arch)

models = {n: load_model(n) for n in MODEL_NAMES}
print('Loaded:', list(models))

## Image loaders (single-image, with paired clean+degraded)

In [ ]:
import pandas as pd
from PIL import Image
from torchvision import transforms as T

TFM = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                 T.ToTensor(),
                 T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def load_tensor(path):
    return TFM(Image.open(path).convert('RGB')).unsqueeze(0).to(DEVICE)

# Pick the same N images we'll explain across every condition
test_ids = pd.read_csv(PHASE2_DIR / 'metrics' / 'test_ids.csv')['id_code'].tolist()
rng = pd.Series(test_ids).sample(N_EXPLAIN, random_state=42)
EXPLAIN_IDS = rng.tolist()
labels_df = pd.read_csv(PRISTINE_CSV).set_index('id_code')['diagnosis']
print('Explaining', len(EXPLAIN_IDS), 'images')

## XAI 1 — Grad-CAM (CNNs)

In [ ]:
import numpy as np
import torch.nn.functional as F

def gradcam_heatmap(model, x, target_class=None):
    layer = get_xai_target_layer(model)
    acts, grads = {}, {}
    h1 = layer.register_forward_hook(lambda _, __, o: acts.update(v=o.detach()))
    h2 = layer.register_full_backward_hook(lambda _, gi, go: grads.update(v=go[0].detach()))
    try:
        model.zero_grad(set_to_none=True)
        x = x.clone().requires_grad_(True)
        logits = model(x)
        if target_class is None:
            target_class = int(logits.argmax(1).item())
        logits[0, target_class].backward()
        a, g = acts['v'][0], grads['v'][0]
        w = g.mean(dim=(1, 2))
        cam = torch.relu((w[:, None, None] * a).sum(0))
        cam = cam - cam.min()
        if cam.max() > 0: cam /= cam.max()
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode='bilinear', align_corners=False)
        return cam.squeeze().cpu().numpy()
    finally:
        h1.remove(); h2.remove()

## XAI 2 — Attention rollout (ViT only)

In [ ]:
@torch.no_grad()
def attention_rollout(model, x):
    attns = []
    handles = []
    for blk in model.blocks:
        attn_module = blk.attn
        def make_hook(store, mod):
            def _h(module, inp, out):
                B, N, C = inp[0].shape
                qkv = mod.qkv(inp[0]).reshape(B, N, 3, mod.num_heads, C // mod.num_heads).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv[0], qkv[1], qkv[2]
                a = (q @ k.transpose(-2, -1)) * mod.scale
                store.append(a.softmax(dim=-1).detach())
            return _h
        handles.append(attn_module.register_forward_hook(make_hook(attns, attn_module)))
    try:
        _ = model(x)
    finally:
        for h in handles: h.remove()
    result = None
    for a in attns:
        a = a.mean(dim=1)
        a = a + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(dim=-1, keepdim=True)
        result = a if result is None else a @ result
    cls_to_patch = result[0, 0, 1:]
    side = int(cls_to_patch.numel() ** 0.5)
    heat = cls_to_patch.reshape(side, side)
    heat = F.interpolate(heat[None, None], size=x.shape[-2:], mode='bilinear', align_corners=False)
    heat = heat.squeeze().cpu().numpy()
    if heat.max() > heat.min():
        heat = (heat - heat.min()) / (heat.max() - heat.min())
    return heat

## XAI 3 — KernelSHAP on superpixels (model-agnostic)

In [ ]:
import shap
from skimage.segmentation import slic

def shap_heatmap(model, x, target_class=None, n_samples=SHAP_N_SAMPLES, n_segments=40):
    img = x.squeeze(0).permute(1, 2, 0).cpu().numpy()
    seg = slic(img, n_segments=n_segments, compactness=10, start_label=0)

    def f(masks):
        out = np.zeros((masks.shape[0], NUM_CLASSES), dtype=np.float32)
        for i, m in enumerate(masks):
            keep = np.isin(seg, np.where(m == 1)[0])
            masked = img.copy(); masked[~keep] = img.mean(axis=(0, 1))
            t = torch.tensor(masked).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
            with torch.no_grad():
                out[i] = torch.softmax(model(t), 1).cpu().numpy()[0]
        return out

    n_super = int(seg.max() + 1)
    explainer = shap.KernelExplainer(f, np.zeros((1, n_super)))
    sv = explainer.shap_values(np.ones((1, n_super)), nsamples=n_samples)
    if target_class is None:
        with torch.no_grad():
            target_class = int(model(x).argmax(1).item())
    sv = sv[target_class][0]
    heat = np.zeros(seg.shape, dtype=np.float32)
    for sid, val in enumerate(sv):
        heat[seg == sid] = val
    if heat.max() > heat.min():
        heat = (heat - heat.min()) / (heat.max() - heat.min())
    return heat

## Fidelity / Stability / Localization metrics

In [ ]:
from scipy.stats import spearmanr

@torch.no_grad()
def _prob_for_class(model, x, cls):
    return torch.softmax(model(x), 1)[0, cls].item()

def deletion_auc(model, x, heatmap, target_class, steps=DELETION_STEPS):
    """Lower is better — model probability drops as top-k pixels are removed."""
    H, W = heatmap.shape
    order = np.argsort(-heatmap.flatten())
    chunk = max(1, len(order) // steps)
    probs = []
    x_work = x.clone()
    mean_pix = x_work.mean(dim=(2, 3), keepdim=True)
    for s in range(steps + 1):
        idx = order[: s * chunk]
        rows, cols = np.unravel_index(idx, (H, W))
        x_mod = x_work.clone()
        x_mod[0, :, rows, cols] = mean_pix.expand_as(x_work)[0, :, rows, cols]
        probs.append(_prob_for_class(model, x_mod, target_class))
    return float(np.trapz(probs, dx=1 / steps))

def insertion_auc(model, x, heatmap, target_class, steps=DELETION_STEPS):
    """Higher is better — start from a blurred image and re-insert top-k pixels."""
    H, W = heatmap.shape
    order = np.argsort(-heatmap.flatten())
    chunk = max(1, len(order) // steps)
    base = torch.nn.functional.avg_pool2d(x, kernel_size=15, stride=1, padding=7)
    probs = []
    for s in range(steps + 1):
        idx = order[: s * chunk]
        rows, cols = np.unravel_index(idx, (H, W))
        x_mod = base.clone()
        x_mod[0, :, rows, cols] = x[0, :, rows, cols]
        probs.append(_prob_for_class(model, x_mod, target_class))
    return float(np.trapz(probs, dx=1 / steps))

def stability_spearman(h_clean, h_degraded):
    rho, _ = spearmanr(h_clean.flatten(), h_degraded.flatten())
    return float(rho) if rho == rho else 0.0

def localization_iou(heatmap, lesion_mask, percentile=80):
    """IoU of top-percentile heatmap pixels with the binary lesion mask."""
    if lesion_mask is None: return float('nan')
    thr = np.percentile(heatmap, percentile)
    pred = heatmap >= thr
    inter = np.logical_and(pred, lesion_mask).sum()
    union = np.logical_or(pred, lesion_mask).sum()
    return float(inter / union) if union > 0 else float('nan')

## Run the full XAI benchmark

For every (image, model, method, condition) we record fidelity + stability + localization. This is the heaviest cell — expect 20–60 min depending on N_EXPLAIN.

In [ ]:
from tqdm.auto import tqdm

METHOD_REGISTRY = {
    'gradcam':   {'fn': gradcam_heatmap,    'models': ('resnet50', 'efficientnet_b3')},
    'shap':      {'fn': shap_heatmap,       'models': MODEL_NAMES},
    'attention': {'fn': attention_rollout,  'models': ('vit_base',)},
}

# Lesion masks: optional. Drop binary masks at /content/data/lesion_masks/<id>.png if available.
MASK_DIR = LOCAL_ROOT / 'lesion_masks'
def load_mask(id_code):
    p = MASK_DIR / f'{id_code}.png'
    if not p.exists(): return None
    m = np.array(Image.open(p).convert('L').resize((IMAGE_SIZE, IMAGE_SIZE)))
    return (m > 127)

rows = []
for method_name, spec in METHOD_REGISTRY.items():
    fn = spec['fn']
    for model_name in spec['models']:
        model = models[model_name]
        print(f'\n[{method_name} | {model_name}]')
        for id_code in tqdm(EXPLAIN_IDS, desc=f'{method_name}/{model_name}'):
            label = int(labels_df[id_code])
            mask  = load_mask(id_code)
            x_clean = load_tensor(APTOS_IMAGES / f'{id_code}.png')
            try:
                h_clean = fn(model, x_clean, target_class=label)
            except Exception as e:
                print(f'  ! {id_code}: {e}'); continue
            rows.append({'id_code': id_code, 'model': model_name, 'method': method_name,
                         'degradation': 'clean', 'level': 'none',
                         'deletion_auc':  deletion_auc(model, x_clean, h_clean, label),
                         'insertion_auc': insertion_auc(model, x_clean, h_clean, label),
                         'stability':     1.0,  # by definition vs itself
                         'iou':           localization_iou(h_clean, mask)})
            for k in DEGRADATION_TYPES:
                for l in DEGRADATION_LEVELS:
                    src = DEGRADED_DIR / k / l / f'{id_code}.png'
                    if not src.exists(): continue
                    x_deg = load_tensor(src)
                    try:
                        h_deg = fn(model, x_deg, target_class=label)
                    except Exception as e:
                        print(f'  ! {id_code}/{k}/{l}: {e}'); continue
                    rows.append({'id_code': id_code, 'model': model_name, 'method': method_name,
                                 'degradation': k, 'level': l,
                                 'deletion_auc':  deletion_auc(model, x_deg, h_deg, label),
                                 'insertion_auc': insertion_auc(model, x_deg, h_deg, label),
                                 'stability':     stability_spearman(h_clean, h_deg),
                                 'iou':           localization_iou(h_deg, mask)})

xai_df = pd.DataFrame(rows)
xai_csv = PHASE_DIR / 'metrics' / 'xai_results.csv'
xai_df.to_csv(xai_csv, index=False)
print('\nSaved:', xai_csv)
xai_df.head()

## Aggregated metric tables (mean ± std)

In [ ]:
for metric in ('deletion_auc', 'insertion_auc', 'stability', 'iou'):
    g = xai_df.groupby(['model', 'method', 'degradation', 'level'])[metric]
    summary = g.agg(['mean', 'std', 'count']).round(4).reset_index()
    out = PHASE_DIR / 'metrics' / f'summary_{metric}.csv'
    summary.to_csv(out, index=False)
    print('Saved:', out)

## Plots — XAI metric vs degradation severity

In [ ]:
import matplotlib.pyplot as plt

level_order = ['none', 'low', 'mid', 'high']

for metric in ('deletion_auc', 'insertion_auc', 'stability', 'iou'):
    for kind in DEGRADATION_TYPES:
        sub = xai_df[(xai_df['degradation'].isin([kind, 'clean']))].copy()
        sub['level'] = pd.Categorical(sub['level'], categories=level_order, ordered=True)
        if sub[metric].dropna().empty: continue
        g = sub.groupby(['model', 'method', 'level'])[metric].mean().reset_index()
        fig, ax = plt.subplots(figsize=(7, 4.5))
        for (m, mt), grp in g.groupby(['model', 'method']):
            ax.plot(grp['level'], grp[metric], marker='o', label=f'{m}/{mt}')
        ax.set_title(f'{metric} vs {kind} severity')
        ax.set_xlabel('Degradation level'); ax.set_ylabel(metric)
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        plt.tight_layout()
        out = PHASE_DIR / 'plots' / f'{metric}_vs_{kind}.png'
        plt.savefig(out, dpi=150); plt.show()

## Qualitative figure — clean vs (high) degraded heatmaps for one example

In [ ]:
import matplotlib.pyplot as plt

def overlay(img_pil, heat, alpha=0.45):
    arr = np.array(img_pil.resize((IMAGE_SIZE, IMAGE_SIZE))).astype(np.float32) / 255
    cmap = plt.get_cmap('jet')
    h_rgb = cmap(heat)[..., :3]
    return (1 - alpha) * arr + alpha * h_rgb

demo_id = EXPLAIN_IDS[0]
label   = int(labels_df[demo_id])
fig, axes = plt.subplots(len(MODEL_NAMES), 4, figsize=(13, 3.2 * len(MODEL_NAMES)))

src_clean = APTOS_IMAGES / f'{demo_id}.png'
img_clean = Image.open(src_clean).convert('RGB')
x_clean = load_tensor(src_clean)

for r, model_name in enumerate(MODEL_NAMES):
    method = 'attention' if model_name == 'vit_base' else 'gradcam'
    fn = METHOD_REGISTRY[method]['fn']
    h_clean = fn(models[model_name], x_clean, target_class=label)

    axes[r, 0].imshow(img_clean.resize((IMAGE_SIZE, IMAGE_SIZE)))
    axes[r, 0].set_title(f'{model_name} — clean image' if r == 0 else 'clean image')
    axes[r, 1].imshow(overlay(img_clean, h_clean))
    axes[r, 1].set_title(f'{method} on clean')

    src_deg = DEGRADED_DIR / 'blur' / 'high' / f'{demo_id}.png'
    img_deg = Image.open(src_deg).convert('RGB')
    x_deg = load_tensor(src_deg)
    h_deg = fn(models[model_name], x_deg, target_class=label)
    axes[r, 2].imshow(img_deg.resize((IMAGE_SIZE, IMAGE_SIZE)))
    axes[r, 2].set_title('blur-high image')
    axes[r, 3].imshow(overlay(img_deg, h_deg))
    axes[r, 3].set_title(f'{method} on blur-high')

    for ax in axes[r]: ax.axis('off')

plt.tight_layout()
out = PHASE_DIR / 'samples' / f'qualitative_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', out)

---
**Phase 3 complete.** Proceed to `04_phase4_genai_enhancement.ipynb`.